# Comparación de hilos y multiprocesamiento en Python

Notebook preparado para **Google Colab**. Súbelo desde **Archivo > Subir notebook** y ejecuta las celdas en orden. La celda 4 también incluye el ejemplo completo en un solo bloque.

Las pruebas de multiprocesamiento usan `fork`, disponible en el entorno Linux de Colab, para que los procesos puedan ejecutar funciones definidas en una celda interactiva.

## Celda 1: Instalar dependencia

In [13]:
%pip install -q psutil


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Celda 2: Comparación de tiempo entre hilos y multiprocesamiento

In [14]:
import multiprocessing
import queue
import threading
import time

ITERACIONES = 10_100_000
NUM_TAREAS = 2
RESULTADO_ESPERADO = ITERACIONES * (ITERACIONES - 1) // 2

def calc():
    total = 0
    for i in range(ITERACIONES):
        total += i
    return total

def _ejecutar_hilo(funcion, resultados, errores, indice):
    try:
        resultados[indice] = funcion()
    except Exception as error:
        errores[indice] = error

def prueba_hilos(funcion):
    resultados = [None] * NUM_TAREAS
    errores = [None] * NUM_TAREAS
    hilos = [
        threading.Thread(target=_ejecutar_hilo, args=(funcion, resultados, errores, i))
        for i in range(NUM_TAREAS)
    ]
    for hilo in hilos:
        hilo.start()
    for hilo in hilos:
        hilo.join()
    for indice, error in enumerate(errores):
        if error is not None:
            raise RuntimeError(f"Falló el hilo {indice + 1}.") from error
    return resultados

def _ejecutar_proceso(funcion, resultados):
    resultados.put(funcion())

def prueba_multiprocesos(funcion):
    if "fork" not in multiprocessing.get_all_start_methods():
        raise RuntimeError("Se requiere el método 'fork', disponible en Google Colab/Linux.")
    contexto = multiprocessing.get_context("fork")
    cola = contexto.Queue()
    procesos = [
        contexto.Process(target=_ejecutar_proceso, args=(funcion, cola))
        for _ in range(NUM_TAREAS)
    ]
    iniciados = []
    try:
        for proceso in procesos:
            proceso.start()
            iniciados.append(proceso)
        for proceso in iniciados:
            proceso.join()
        fallidos = [proceso for proceso in iniciados if proceso.exitcode != 0]
        if fallidos:
            detalle = ", ".join(f"PID {p.pid}: código {p.exitcode}" for p in fallidos)
            raise RuntimeError(f"Falló la ejecución de un proceso ({detalle}).")
        try:
            return [cola.get(timeout=30) for _ in iniciados]
        except queue.Empty as error:
            raise RuntimeError("Un proceso terminó sin devolver su resultado.") from error
    finally:
        for proceso in iniciados:
            if proceso.is_alive():
                proceso.terminate()
            proceso.join()
        cola.close()
        cola.join_thread()

inicio = time.perf_counter()
resultados_hilos = prueba_hilos(calc)
tiempo_hilos = time.perf_counter() - inicio
if resultados_hilos != [RESULTADO_ESPERADO] * NUM_TAREAS:
    raise RuntimeError(f"Resultado incorrecto en hilos: {resultados_hilos}")
print(f"Tiempo de ejecución de los hilos: {tiempo_hilos:.3f} s")

inicio = time.perf_counter()
resultados_procesos = prueba_multiprocesos(calc)
tiempo_procesos = time.perf_counter() - inicio
if resultados_procesos != [RESULTADO_ESPERADO] * NUM_TAREAS:
    raise RuntimeError(f"Resultado incorrecto en multiprocesos: {resultados_procesos}")
print(f"Tiempo de ejecución de los procesos: {tiempo_procesos:.3f} s")
print(f"Resultado correcto por tarea: {RESULTADO_ESPERADO}")

Tiempo de ejecución de los hilos: 0.873 s
Tiempo de ejecución de los procesos: 0.431 s
Resultado correcto por tarea: 51004994950000


## Celda 3: Verificar PID, núcleo y GIL con `psutil`

In [15]:
import os
import psutil

def calc2():
    pid = os.getpid()
    try:
        cpu = psutil.Process(pid).cpu_num()
    except (psutil.Error, NotImplementedError, AttributeError) as error:
        cpu = f"no disponible ({type(error).__name__})"
    print(f"[PID={pid} | CPU={cpu}] Iniciando cálculo ...", flush=True)
    total = 0
    for i in range(ITERACIONES):
        total += i
    print(f"[PID={pid} | CPU={cpu}] Finalizando cálculo ...", flush=True)
    return total

print("--- Prueba con hilos ---")
resultados_hilos = prueba_hilos(calc2)
if resultados_hilos != [RESULTADO_ESPERADO] * NUM_TAREAS:
    raise RuntimeError(f"Resultado incorrecto en hilos: {resultados_hilos}")

print("\n--- Prueba con multiprocesos ---")
resultados_procesos = prueba_multiprocesos(calc2)
if resultados_procesos != [RESULTADO_ESPERADO] * NUM_TAREAS:
    raise RuntimeError(f"Resultado incorrecto en multiprocesos: {resultados_procesos}")
print("Ambas pruebas terminaron con resultados correctos.")

--- Prueba con hilos ---
[PID=91789 | CPU=no disponible (AttributeError)] Iniciando cálculo ...
[PID=91789 | CPU=no disponible (AttributeError)] Iniciando cálculo ...
[PID=91789 | CPU=no disponible (AttributeError)] Finalizando cálculo ...
[PID=91789 | CPU=no disponible (AttributeError)] Finalizando cálculo ...

--- Prueba con multiprocesos ---
[PID=93713 | CPU=no disponible (AttributeError)] Iniciando cálculo ...
[PID=93714 | CPU=no disponible (AttributeError)] Iniciando cálculo ...
[PID=93713 | CPU=no disponible (AttributeError)] Finalizando cálculo ...
[PID=93714 | CPU=no disponible (AttributeError)] Finalizando cálculo ...
Ambas pruebas terminaron con resultados correctos.


## Celda 4: Bloque completo unificado (imagen 5)

Esta celda es autocontenida: ejecuta la comparación completa con PID, CPU, hilos, multiprocesos y medición de tiempo. Puedes ejecutarla después de instalar `psutil`, sin ejecutar las celdas 2 y 3.

In [16]:
import multiprocessing
import os
import psutil
import queue
import threading
import time

ITERACIONES = 10_100_000
NUM_TAREAS = 2
RESULTADO_ESPERADO = ITERACIONES * (ITERACIONES - 1) // 2

def calc2():
    pid = os.getpid()
    try:
        cpu = psutil.Process(pid).cpu_num()
    except (psutil.Error, NotImplementedError, AttributeError) as error:
        cpu = f"no disponible ({type(error).__name__})"
    print(f"[PID={pid} | CPU={cpu}] Iniciando cálculo ...", flush=True)
    total = 0
    for i in range(ITERACIONES):
        total += i
    print(f"[PID={pid} | CPU={cpu}] Finalizando cálculo ...", flush=True)
    return total

def _ejecutar_hilo(funcion, resultados, errores, indice):
    try:
        resultados[indice] = funcion()
    except Exception as error:
        errores[indice] = error

def prueba_hilos():
    resultados = [None] * NUM_TAREAS
    errores = [None] * NUM_TAREAS
    hilos = [
        threading.Thread(target=_ejecutar_hilo, args=(calc2, resultados, errores, i))
        for i in range(NUM_TAREAS)
    ]
    for hilo in hilos:
        hilo.start()
    for hilo in hilos:
        hilo.join()
    for indice, error in enumerate(errores):
        if error is not None:
            raise RuntimeError(f"Falló el hilo {indice + 1}.") from error
    return resultados

def _ejecutar_proceso(funcion, resultados):
    resultados.put(funcion())

def prueba_multiprocesos():
    if "fork" not in multiprocessing.get_all_start_methods():
        raise RuntimeError("Se requiere el método 'fork', disponible en Google Colab/Linux.")
    contexto = multiprocessing.get_context("fork")
    cola = contexto.Queue()
    procesos = [
        contexto.Process(target=_ejecutar_proceso, args=(calc2, cola))
        for _ in range(NUM_TAREAS)
    ]
    iniciados = []
    try:
        for proceso in procesos:
            proceso.start()
            iniciados.append(proceso)
        for proceso in iniciados:
            proceso.join()
        fallidos = [proceso for proceso in iniciados if proceso.exitcode != 0]
        if fallidos:
            detalle = ", ".join(f"PID {p.pid}: código {p.exitcode}" for p in fallidos)
            raise RuntimeError(f"Falló la ejecución de un proceso ({detalle}).")
        try:
            return [cola.get(timeout=30) for _ in iniciados]
        except queue.Empty as error:
            raise RuntimeError("Un proceso terminó sin devolver su resultado.") from error
    finally:
        for proceso in iniciados:
            if proceso.is_alive():
                proceso.terminate()
            proceso.join()
        cola.close()
        cola.join_thread()

print("Hilos")
inicio = time.perf_counter()
resultados_hilos = prueba_hilos()
tiempo_hilos = time.perf_counter() - inicio
if resultados_hilos != [RESULTADO_ESPERADO] * NUM_TAREAS:
    raise RuntimeError(f"Resultado incorrecto en hilos: {resultados_hilos}")
print(f"El tiempo de ejecución de los hilos: {tiempo_hilos:.3f} s")

print("\nMultiprocesos")
inicio = time.perf_counter()
resultados_procesos = prueba_multiprocesos()
tiempo_procesos = time.perf_counter() - inicio
if resultados_procesos != [RESULTADO_ESPERADO] * NUM_TAREAS:
    raise RuntimeError(f"Resultado incorrecto en multiprocesos: {resultados_procesos}")
print(f"El tiempo de ejecución de los procesos: {tiempo_procesos:.3f} s")
print(f"Resultado correcto por tarea: {RESULTADO_ESPERADO}")

Hilos
[PID=91789 | CPU=no disponible (AttributeError)] Iniciando cálculo ...
[PID=91789 | CPU=no disponible (AttributeError)] Iniciando cálculo ...
[PID=91789 | CPU=no disponible (AttributeError)] Finalizando cálculo ...
[PID=91789 | CPU=no disponible (AttributeError)] Finalizando cálculo ...
El tiempo de ejecución de los hilos: 0.829 s

Multiprocesos
[PID=93719 | CPU=no disponible (AttributeError)] Iniciando cálculo ...
[PID=93720 | CPU=no disponible (AttributeError)] Iniciando cálculo ...
[PID=93719 | CPU=no disponible (AttributeError)] Finalizando cálculo ...
[PID=93720 | CPU=no disponible (AttributeError)] Finalizando cálculo ...
El tiempo de ejecución de los procesos: 0.431 s
Resultado correcto por tarea: 51004994950000


### Nota sobre los resultados

`cpu_num()` muestra el núcleo en el que se observó el proceso, pero no fija una tarea a un núcleo específico. Los hilos comparten el intérprete y el GIL limita el paralelismo de código Python intensivo en CPU. Los procesos usan intérpretes separados. Los tiempos dependen del entorno de Colab y de la carga del sistema.

## Celda de texto: uso de CPU y memoria

El siguiente bloque ejecuta suma, resta, multiplicación y división en procesos separados. Cada proceso muestra su PID, porcentaje de CPU medido durante la operación y memoria residente (RSS) utilizada. También informa qué porcentaje de la memoria total del sistema representa esa memoria.

In [17]:
import multiprocessing as mp
import os
import time
import psutil
import queue

def suma(a, b):
    return a + b

def resta(a, b):
    return a - b

def multiplicacion(a, b):
    return a * b

def division(a, b):
    if b == 0:
        raise ZeroDivisionError("No se puede dividir entre cero.")
    return a / b

def ejecutar_operacion(nombre, simbolo, funcion, a, b, resultados):
    pid = os.getpid()
    proceso = psutil.Process(pid)
    cpu_inicio = proceso.cpu_times()
    tiempo_inicio = time.perf_counter()
    resultado = funcion(a, b)
    tiempo_transcurrido = time.perf_counter() - tiempo_inicio
    cpu_fin = proceso.cpu_times()
    tiempo_cpu = (cpu_fin.user + cpu_fin.system) - (cpu_inicio.user + cpu_inicio.system)
    cpu_porcentaje = (tiempo_cpu / tiempo_transcurrido * 100) if tiempo_transcurrido else 0.0
    memoria = proceso.memory_info().rss
    memoria_porcentaje = proceso.memory_percent()
    print(
        f"{nombre}: {a} {simbolo} {b} = {resultado} | PID: {pid} "
        f"| CPU: {cpu_porcentaje:.2f}% | Memoria RSS: {memoria / 1024:.2f} KB "
        f"({memoria_porcentaje:.2f}% de la memoria del sistema)",
        flush=True,
    )
    resultados.put((nombre, resultado))

num1, num2 = 10, 20
operaciones = [
    ("suma", "+", suma),
    ("resta", "-", resta),
    ("multiplicacion", "*", multiplicacion),
    ("division", "/", division),
]

if "fork" not in mp.get_all_start_methods():
    raise RuntimeError("Esta celda requiere el método 'fork', disponible en Google Colab/Linux.")
contexto = mp.get_context("fork")
cola_resultados = contexto.Queue()
procesos = [
    contexto.Process(
        target=ejecutar_operacion,
        args=(nombre, simbolo, funcion, num1, num2, cola_resultados),
    )
    for nombre, simbolo, funcion in operaciones
]
iniciados = []
try:
    for proceso in procesos:
        proceso.start()
        iniciados.append(proceso)
    for proceso in iniciados:
        proceso.join()

    fallidos = [proceso for proceso in iniciados if proceso.exitcode != 0]
    if fallidos:
        detalle = ", ".join(f"PID {proceso.pid}: código {proceso.exitcode}" for proceso in fallidos)
        raise RuntimeError(f"Falló la ejecución de uno o más procesos ({detalle}).")

    try:
        resultados = [cola_resultados.get(timeout=30) for _ in iniciados]
    except queue.Empty as error:
        raise RuntimeError("Un proceso terminó sin devolver el resultado de su operación.") from error
    if len(resultados) != len(operaciones):
        raise RuntimeError(f"Se esperaban {len(operaciones)} resultados y se recibieron {len(resultados)}.")
    print("Los procesos de suma, resta, multiplicación y división han terminado correctamente.")
finally:
    for proceso in iniciados:
        if proceso.is_alive():
            proceso.terminate()
        proceso.join()
    cola_resultados.close()
    cola_resultados.join_thread()

suma: 10 + 20 = 30 | PID: 93806 | CPU: 1748.25% | Memoria RSS: 11504.00 KB (0.07% de la memoria del sistema)
resta: 10 - 20 = -10 | PID: 93807 | CPU: 2165.39% | Memoria RSS: 11504.00 KB (0.07% de la memoria del sistema)multiplicacion: 10 * 20 = 200 | PID: 93808 | CPU: 2006.79% | Memoria RSS: 11568.00 KB (0.07% de la memoria del sistema)

division: 10 / 20 = 0.5 | PID: 93809 | CPU: 205.60% | Memoria RSS: 11552.00 KB (0.07% de la memoria del sistema)
Los procesos de suma, resta, multiplicación y división han terminado correctamente.


# Actividad: Cuestionario sobre multiprocesamiento

### 1. ¿Qué hace el método `spawn`?

Crea un proceso hijo nuevo iniciando otro intérprete de Python. Ese intérprete prepara el entorno e importa el módulo que define el trabajo; luego ejecuta la función objetivo de forma independiente, en vez de clonar el estado de memoria del proceso padre.

### 2. ¿Qué significa iniciar un nuevo intérprete y qué ocurre con las variables globales del padre?

El proceso hijo empieza en un intérprete separado y vuelve a importar el módulo. No recibe automáticamente los valores globales que el padre modificó durante la ejecución. Las variables globales se crean según el código de nivel superior que se ejecuta al importar el módulo; no necesariamente vuelven a un valor por defecto si el propio módulo las inicializa de otra manera.

### 3. ¿Por qué los procesos `spawn` no comparten directamente memoria? Indica una ventaja y una desventaja.

Cada proceso tiene su propio espacio de direcciones virtuales, por lo que las variables de uno no son directamente las variables del otro. **Ventaja:** el aislamiento reduce interferencias accidentales entre procesos. **Desventaja:** se puede duplicar memoria y compartir datos requiere comunicación explícita (IPC), como colas, tuberías o memoria compartida.

### 4. ¿Por qué `spawn` suele tardar más en iniciarse que `fork`?

`spawn` debe iniciar un intérprete, preparar el entorno e importar los módulos necesarios. En sistemas compatibles, `fork` crea un proceso a partir del actual y puede aprovechar *Copy-on-Write* para páginas de memoria; el coste de arranque suele ser menor. El coste concreto depende del sistema y de los módulos que se deban cargar.

### 5. ¿Por qué es importante `if __name__ == "__main__"` con `spawn`? ¿Qué podría ocurrir si se elimina?

Con `spawn`, el hijo importa el módulo principal. El guard evita que al importarlo vuelva a ejecutarse el código que crea procesos. Sin él, puede producirse una creación recursiva de procesos o el error de arranque seguro de `multiprocessing`; por eso la creación de procesos debe protegerse con este bloque.

### 6. Si se crean cuatro procesos con `spawn`, ¿hay un intérprete compartido o varios?

Hay cuatro intérpretes hijos independientes, uno por cada proceso, además del intérprete del proceso padre: cinco procesos/intérpretes en total mientras todos estén activos.

### 7. ¿Qué impacto tiene `spawn` al crear cientos de procesos de corta duración y dónde se nota?

El arranque de intérpretes e importación de módulos puede costar más que el propio trabajo, aumentando el tiempo total. También puede elevar el uso de CPU y memoria. Para muchas tareas pequeñas suele convenir reutilizar un grupo fijo de procesos (*process pool*) en vez de crear un proceso por tarea.

### 8. Compara `spawn` con la creación tradicional de un proceso en un sistema operativo. ¿Qué prepara el sistema y qué permanece independiente?

Con `spawn`, Python solicita al sistema operativo un proceso nuevo e inicia un intérprete de Python en él. El sistema prepara estructuras de proceso (por ejemplo, PID y estado), un espacio de direcciones virtuales, recursos de ejecución y los manejadores que se hereden o configuren. El intérprete y la memoria privada del hijo (pila, heap y variables ordinarias) son independientes del padre. Los recursos compartidos requieren mecanismos explícitos; algunos identificadores pueden heredarse o transferirse según la plataforma y la configuración.

### 9. Explica el funcionamiento de `fork`.

`fork` crea un proceso hijo a partir del proceso actual. El hijo comienza con una copia lógica del estado del padre en el punto de creación; normalmente, la memoria se administra con *Copy-on-Write*, de modo que las páginas se copian cuando alguno de los procesos las modifica. Padre e hijo tienen PID distintos y cambios en variables ordinarias no se convierten en cambios compartidos. `fork` está disponible principalmente en sistemas Unix; el método disponible depende de la plataforma.

### 10. ¿Se puede compartir información entre hilos o procesos? Explica.

- **Hilos:** sí. Los hilos de un mismo proceso comparten el espacio de memoria y pueden acceder a los mismos objetos. Para coordinar escrituras concurrentes se pueden necesitar `Lock` u otras primitivas de sincronización.
- **Procesos:** sí, mediante mecanismos explícitos de IPC, como `multiprocessing.Queue`, `Pipe`, `Value`, `Array` o memoria compartida. La memoria ordinaria de un proceso no se comparte automáticamente con los demás.